# Instalação das bibliotecas necessárias

In [0]:
!pip install feature_engine

In [0]:
!pip install xgboost

In [0]:
!pip install lightgbm

In [0]:
!pip install catboost

# Importações de bibliotecas

In [0]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from feature_engine import imputation
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

import mlflow
import mlflow.sklearn

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn import model_selection
from sklearn.feature_selection import SequentialFeatureSelector

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss
)

from mlflow.models import infer_signature

import os
import logging

# Importação dos dados e Pré-Processamento dos Dados

## Classificação dos Feedbacks
Executar apenas uma vez

In [0]:
#import json
#from pyspark.sql import functions as F

# 1. Rótulos: nome -> descrição (as descrições orientam o LLM; baseadas nas regras definidas)
#labels = {
#    "satisfacao": "The customer expresses satisfaction with the service and does NOT mention deciding to cancel, churn or switch to another provider.",
#    "preco_alto": "Primary reason is the monthly charges being too high or expensive, without mentioning having found a competitor's better offer.",
#    "oferta_concorrente": "Primary reason is having found a better deal, better price or promotional offer from another provider, deciding to switch.",
#    "qualidade_servico": "Primary reason is problems with the service itself: slow speeds, unreliable connection, frequent outages or interruptions.",
#    "mudanca_endereco": "Primary reason is moving to a new location or address (relocation).",
#    "metodo_pagamento": "Primary reason is difficulties or inconvenience with the payment method or billing process (not the price itself).",
#    "motivo_pessoal": "Primary reason is personal circumstances, unforeseen situations or financial constraints, unrelated to the provider.",
#    "outro": "Any other primary reason not covered by the labels above."
#}

# 2. Instruções globais: codificam as regras de classificação
#instructions = (
#    "Inputs are English customer feedback texts from internet/phone service customers. "
#    "Some of them canceled the service or switched providers. "
#    "If the customer expresses satisfaction and does not mention canceling or switching, classify as 'satisfacao'. "
#    "If the text indicates the customer decided to cancel or switch (even when otherwise satisfied), "
#    "classify by the PRIMARY reason stated for that decision. "
#    "If the customer expresses dissatisfaction without canceling, classify the PRIMARY reason of the dissatisfaction. "
#    "When the customer mentions both high prices and having found a better deal elsewhere, prefer 'oferta_concorrente'."
#)

#labels_str = json.dumps(labels).replace("'", "''")
#instructions_str = instructions.replace("'", "''")

# 3. Classificação: uma chamada de LLM por linha, paralelizada pelo Spark
#df_classificado = (
#    spark.table("workspace.default.train")
#    .withColumn(
#        "motivo_feedback",
#        F.expr(f"""
#            ai_classify(
#               CustomerFeedback,
#                '{labels_str}',
#                MAP('version', '2.1', 'instructions', '{instructions_str}')
#            ):response[0].value::STRING
#        """)
#    )
#)

# 4. Persistir para não pagar a classificação novamente a cada execução
#df_classificado.write.mode("overwrite").saveAsTable("workspace.default.train_classificado")

## Carregamento e pré-processamento dos dados

In [0]:
#df_abt = spark.table("workspace.default.train_classificado").toPandas()
df_abt = spark.table("workspace.default.train").toPandas()

In [0]:
df_abt

In [0]:
# ============================================================
# FILTRAGEM DE OBSERVACOES
# ============================================================

def filtrar_observacoes_invalidas(df):
    """
    Exclui observacoes onde simultaneamente:

        tenure == 'unknown'
        AND
        TotalCharges == NULL

    Essa regra nao depende de estatisticas da base e, portanto, pode ser executada antes do split.
    Retorna um novo DataFrame sem modificar o original.
    """

    df = df.copy()

    tenure_unknown = (df["tenure"].astype("string").str.strip().str.lower().eq("unknown"))

    total_charges_null = (df["TotalCharges"].isna())

    mascara_excluir = (tenure_unknown & total_charges_null)

    return (df.loc[~mascara_excluir].copy().reset_index(drop=True))


In [0]:
df_abt = filtrar_observacoes_invalidas(df_abt)
df_abt

In [0]:
# ============================================================
# TRANSFORMAÇÃO DO TARGET PARA NUMÉRICO
# ============================================================

def transformar_target(df, coluna='Churn'):

    """
    Transforma o target em 0 (Churn = yes) ou 1 (Churn = no).
    Retorna um novo DataFrame sem modificar o original.
    """

    df = df.copy()

    df[coluna] = (
        df[coluna]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            'yes': 1,
            'no': 0
        })
    )

    return df


In [0]:
target = 'Churn'

df_abt = transformar_target(df_abt, target)
df_abt

In [0]:
X = df_abt.drop(columns=[target])
y = df_abt[target]

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y,
                                                                    random_state=42,
                                                                    stratify=y,
                                                                    test_size=0.2)

print(f"Tamanho da base Treino: {y_train.shape[0]} ({y_train.shape[0]/y.shape[0]*100:.2f}%). Taxa respoata: ({y_train.mean()*100:.2f}%)")

print(f"Tamanho da base Test: {y_test.shape[0]} ({y_test.shape[0]/y.shape[0]*100:.2f}%). Taxa resposta: ({y_test.mean()*100:.2f}%)")

In [0]:
X_train

In [0]:
y_train

In [0]:
# ============================================================
# TRANSFORMACOES INDEPENDENTES DA BASE
# ============================================================

def transformar_dados(df):
    """
    Executa transformacoes que dependem apenas dos valores existentes na propria observacao.
    Nenhuma estatistica é aprendida a partir da base.
    """

    df = df.copy()

    # --------------------------------------------------------
    # Excluir customerID
    # --------------------------------------------------------
    df = df.drop(columns=['customerID'])

    # --------------------------------------------------------
    # SeniorCitizen: 
    # 1 -> yes
    # 0 -> no
    # --------------------------------------------------------

    mapa_senior = {
        1: 'yes',
        0: 'no'
    }

    df['SeniorCitizen'] = (df['SeniorCitizen'].map(mapa_senior).astype('string'))

    # --------------------------------------------------------
    # Variaveis categoricas para minusculas, à exceção de:
    # CustomerFeedback: preservado para processamento NLP posterior.
    # --------------------------------------------------------

    colunas_categoricas = (df.select_dtypes(include=['object', 'string', 'category']).columns)

    colunas_excluidas = ['CustomerFeedback' ]

    for coluna in colunas_categoricas:
        if coluna not in colunas_excluidas:
            df[coluna] = (df[coluna].astype("string").str.strip().str.lower())

    # --------------------------------------------------------
    # qtd_meses_contrato, de acordo com Contract
    # month-to-month -> 1
    # one year -> 12
    # two year -> 24
    # --------------------------------------------------------

    mapa_contrato = {
        'month-to-month': 1,
        'one year': 12,
        'two year': 24
    }

    df['qtd_meses_contrato'] = (df['Contract'].map(mapa_contrato).astype('Int64'))

    # --------------------------------------------------------
    # tenure_num
    # Se tenure for numerico: tenure_num = tenure
    # Se tenure = unknown: tenure_num = ROUND(TotalCharges / MonthlyCharges)
    # --------------------------------------------------------

    df['tenure_num'] = pd.to_numeric(df['tenure'], errors='coerce')

    mascara_estimar_tenure = (
        df['tenure_num'].isna()
        & df['TotalCharges'].notna()
        & df['MonthlyCharges'].notna()
        & df['MonthlyCharges'].ne(0)
    )

    df.loc[mascara_estimar_tenure, 'tenure_num'] = (
        df.loc[mascara_estimar_tenure, 'TotalCharges']
        / df.loc[mascara_estimar_tenure, 'MonthlyCharges']
    ).round()

    # --------------------------------------------------------
    # Imputacao de TotalCharges
    # TotalCharges = tenure_num * MonthlyCharges
    # --------------------------------------------------------

    mascara_imputar_total = (
        df["TotalCharges"].isna()
        & df["tenure_num"].notna()
        & df["MonthlyCharges"].notna()
    )

    df.loc[mascara_imputar_total, 'TotalCharges'] = (
        df.loc[mascara_imputar_total, 'tenure_num']
        * df.loc[mascara_imputar_total, 'MonthlyCharges']
    )

    # --------------------------------------------------------
    # comprometimentoRenda
    # MonthlyCharges / MonthlyIncome
    # --------------------------------------------------------

    renda_valida = (
        df['MonthlyIncome'].notna()
        & df['MonthlyIncome'].ne(0)
    )

    df['comprometimentoRenda'] = np.where(renda_valida, (df['MonthlyCharges'] / df['MonthlyIncome']), np.nan)

    # --------------------------------------------------------
    # avgMonthlyCharges
    # tenure_num == 0: MonthlyCharges
    # caso contrario: TotalCharges / tenure_num
    # --------------------------------------------------------

    df['avgMonthlyCharges'] = np.where(
        df['tenure_num'].eq(0), df['MonthlyCharges'],
        (df['TotalCharges'] / df['tenure_num'])
    )

    # --------------------------------------------------------
    # evolucaoMonthlyCharges
    # MonthlyCharges / avgMonthlyCharges
    # --------------------------------------------------------

    avg_valido = (
        df["avgMonthlyCharges"].notna()
        & df["avgMonthlyCharges"].ne(0)
    )

    df['evolucaoMonthlyCharges'] = np.where(avg_valido, (df['MonthlyCharges'] / df['avgMonthlyCharges']), np.nan)

    # --------------------------------------------------------
    # Excluir tenure, CustomerFeedback e motivo_feedback
    # --------------------------------------------------------
    #df = df.drop(columns=['tenure', 'CustomerFeedback', 'motivo_feedback'])
    df = df.drop(columns=['tenure', 'CustomerFeedback'])

    return df

In [0]:
# ============================================================
# TRANSFORMACAO DEPENDENTE DO TREINO
# ============================================================

class MedianaTenureContrato(
    BaseEstimator,
    TransformerMixin
):
    """
    Aprende no treino a mediana de tenure_num para cada categoria de Contract.

    As medianas aprendidas sao reutilizadas no teste e posteriormente nos dados futuros.
    """

    def fit(self, X, y=None):

        self.medianas_ = (
            X
            .groupby("Contract")["tenure_num"]
            .median()
            .to_dict()
        )

        # Mediana global utilizada como fallback para um  Contract desconhecido em dados futuros.

        self.mediana_global_ = (
            X["tenure_num"]
            .median()
        )

        return self

    def transform(self, X):

        X = X.copy()

        X["mediana_tenure_contrato"] = (
            X["Contract"]
            .map(self.medianas_)
            .fillna(
                self.mediana_global_
            )
        )

        return X

In [0]:
scaler = MinMaxScaler()

In [0]:
# One Hot Encoding das variáveis categóricas da ABT

cat_features = [
    'gender',
    'SeniorCitizen',
    'Partner',
    'Dependents',
    'PhoneService',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaperlessBilling',
    'PaymentMethod',
    #'motivo_feedback'
]

onehot = ColumnTransformer(
    transformers=[
        ('onehot_cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ],
    verbose_feature_names_out=False
)

# Pipeline

In [0]:
# Regressão Logística

pipeline_logistica = Pipeline(steps=[
    ('transformacoes_independentes', FunctionTransformer(transformar_dados, validate=False)),
    ('mediana_tenure_contrato', MedianaTenureContrato()),
    ('onehot', onehot),
    ('scaler', scaler),
    #('selecao_features', FunctionTransformer(selecionar_features_modelo, validate=False)),
    ('modelo', LogisticRegression(max_iter=3000, random_state=42))
])

grid_logistica = {
    "modelo__C": [0.01, 0.1, 1, 10],
    "modelo__class_weight": [None, "balanced"],
    "modelo__solver": ["liblinear", "lbfgs"]
}

In [0]:
# Regressão Logística com Seleção de Variáveis

cv_selecao = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

modelo_selecao = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    solver="liblinear",
    random_state=42
)

seletor_stepwise = SequentialFeatureSelector(
    estimator=modelo_selecao,
    n_features_to_select='auto',
    direction="forward",
    scoring="roc_auc",
    cv=cv_selecao,
    n_jobs=-1
)

pipeline_logistica_selecao = Pipeline(steps=[
    ('transformacoes_independentes', FunctionTransformer(transformar_dados, validate=False)),
    ('mediana_tenure_contrato', MedianaTenureContrato()),
    ('onehot', onehot),
    ('scaler', scaler),
    ('seletor', seletor_stepwise),
    ('modelo', LogisticRegression(max_iter=3000, random_state=42))
])

grid_logistica_selecao = {
    "modelo__C": [0.01, 0.1, 1, 10],
    "modelo__class_weight": [None, "balanced"],
    "modelo__solver": ["liblinear", "lbfgs"]
}


In [0]:
# Random Forest

pipeline_random_forest = Pipeline([
    ('transformacoes_independentes', FunctionTransformer(transformar_dados, validate=False)),
    ('mediana_tenure_contrato', MedianaTenureContrato()),
    ('onehot', onehot),
    ("modelo", RandomForestClassifier(random_state=42, n_jobs=-1))
])

grid_random_forest = {
    "modelo__n_estimators": [350, 1000],
    "modelo__max_depth": [5, 15],
    "modelo__min_samples_split": [2, 15],
    "modelo__min_samples_leaf": [3, 8, 20],
    "modelo__max_features": ["sqrt", 0.5, None],
    "modelo__max_samples": [0.5, 1.0],
    "modelo__class_weight": [None, "balanced"]
}

In [0]:
# Hist Gradient Boosting

pipeline_hist_gradient = Pipeline([
    ('transformacoes_independentes', FunctionTransformer(transformar_dados, validate=False)),
    ('mediana_tenure_contrato', MedianaTenureContrato()),
    ('onehot', onehot),
    ("modelo", HistGradientBoostingClassifier(random_state=42, early_stopping=True))
])

grid_hist_gradient = {
    "modelo__learning_rate": [0.03, 0.05, 0.1],
    "modelo__max_iter": [200, 400, 600],
    "modelo__max_leaf_nodes": [15, 31, 63],
    "modelo__min_samples_leaf": [10, 20, 50],
    "modelo__l2_regularization": [0.0, 0.1, 1.0]
}

In [0]:
# XG Boost

pipeline_xgboost = Pipeline([
    ('transformacoes_independentes', FunctionTransformer(transformar_dados, validate=False)),
    ('mediana_tenure_contrato', MedianaTenureContrato()),
    ('onehot', onehot),
    ("modelo", XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ))
])

grid_xgboost = {
    "modelo__n_estimators": [300, 600],
    "modelo__learning_rate": [0.03, 0.1],
    "modelo__max_depth": [3, 6],
    "modelo__min_child_weight": [1, 5],
    "modelo__subsample": [0.8, 1.0],
    "modelo__colsample_bytree": [0.8, 1.0],
    "modelo__reg_alpha": [0.0, 0.1],
    "modelo__reg_lambda": [1.0, 5.0]
}

In [0]:
# Light GBM

pipeline_lightgbm = Pipeline([
    ('transformacoes_independentes', FunctionTransformer(transformar_dados, validate=False)),
    ('mediana_tenure_contrato', MedianaTenureContrato()),
    ('onehot', onehot),
    ("modelo", LGBMClassifier(
        objective="binary",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

grid_lightgbm = {
    "modelo__n_estimators": [300, 600],
    "modelo__learning_rate": [0.03, 0.1],
    #"modelo__num_leaves": [31, 63],
    "modelo__max_depth": [6, 10],
    #"modelo__min_child_samples": [20, 50],
    #"modelo__subsample": [0.8, 1.0],
    #"modelo__colsample_bytree": [0.8, 1.0],
    #"modelo__reg_alpha": [0.0, 0.1],
    #"modelo__reg_lambda": [0.0, 1.0]
}

In [0]:
# Cat Boost

pipeline_catboost = Pipeline([
    ('transformacoes_independentes', FunctionTransformer(transformar_dados, validate=False)),
    ('mediana_tenure_contrato', MedianaTenureContrato()),
    ('onehot', onehot),
    ("modelo", CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1
    ))
])

grid_catboost = {
    "modelo__iterations": [300, 600],
    "modelo__learning_rate": [0.03, 0.1],
    "modelo__depth": [4, 8],
    #"modelo__l2_leaf_reg": [3, 10],
    #"modelo__random_strength": [0.0, 1.0]
}

In [0]:
# Redes Neurais - MLP Classifier

pipeline_mlp = Pipeline([
    ('transformacoes_independentes', FunctionTransformer(transformar_dados, validate=False)),
    ('mediana_tenure_contrato', MedianaTenureContrato()),
    ('onehot', onehot),
    ('scaler', scaler),
    ('modelo', MLPClassifier(
        random_state=42,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20
    ))
])

grid_mlp = {
    "modelo__hidden_layer_sizes": [(64,), (128, 64)],
    "modelo__activation": ["relu"],
    "modelo__alpha": [0.0001, 0.001],
    "modelo__learning_rate_init": [0.0005, 0.001],
    "modelo__batch_size": [128, 256]
}

In [0]:
# Dicionário dos Modelos

modelos = {
    "regressao_logistica": {
        "pipeline": pipeline_logistica,
        "param_grid": grid_logistica
    } ,
    "regressao_logistica_selecao_var": {
        "pipeline": pipeline_logistica_selecao,
        "param_grid": grid_logistica_selecao
    },
    "random_forest": {
        "pipeline": pipeline_random_forest,
        "param_grid": grid_random_forest
    },
    "hist_gradient_boosting": {
        "pipeline": pipeline_hist_gradient,
        "param_grid": grid_hist_gradient
    },
    "xgboost": {
        "pipeline": pipeline_xgboost,
        "param_grid": grid_xgboost
    },
    "lightgbm": {
        "pipeline": pipeline_lightgbm,
        "param_grid": grid_lightgbm
    },
    "catboost": {
        "pipeline": pipeline_catboost,
        "param_grid": grid_catboost
    },
    "mlp": {
        "pipeline": pipeline_mlp,
        "param_grid": grid_mlp
    }
}

In [0]:
# Configuração do GridSearch

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

In [0]:
# Configuração do MlFlow
os.environ["MLFLOW_DATABRICKS_TAGS_TO_SKIP"] = "extraContext"
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

# VINCULA O SCRIPT PYTHON COM O EXPERIMENTO DO MLFLOW
mlflow.set_experiment(experiment_id=3964855821008334)

mlflow.sklearn.autolog(
    log_models=False,
    max_tuning_runs=20
)

# Treinamento dos modelos

In [0]:
resultados = []
melhores_estimadores = {}

melhor_roc_auc = -np.inf
melhor_modelo_validacao = None
melhor_nome_modelo = None
melhor_run_id = None
melhores_parametros = None

for nome_modelo, configuracao in modelos.items():

    with mlflow.start_run(
        #run_name=f"selecao_{nome_modelo}"
        run_name=nome_modelo
    ) as run:

        grid = GridSearchCV(
            estimator=configuracao["pipeline"],
            param_grid=configuracao["param_grid"],
            scoring=scoring,
            refit="roc_auc",
            cv=cv,
            n_jobs=-1,
            verbose=2,
            return_train_score=True,
            error_score="raise"
        )

        grid.fit(X_train, y_train)

        melhor_estimador = grid.best_estimator_
        melhores_estimadores[nome_modelo] = melhor_estimador

        # Treino
        # Probabilidade da classe positiva
        y_train_prob = melhor_estimador.predict_proba(X_train)[:, 1]
        # Predição usando threshold padrão
        y_train_pred = (y_train_prob >= 0.5).astype(int)

        # Teste
        # Probabilidade da classe positiva
        y_test_prob = melhor_estimador.predict_proba(X_test)[:, 1]
        # Predição usando threshold padrão
        y_test_pred = (y_test_prob >= 0.5).astype(int)

        metricas = {
            "train_roc_auc": roc_auc_score(y_train, y_train_prob),
            "train_average_precision": average_precision_score(y_train, y_train_prob),
            "train_log_loss": log_loss(y_train, y_train_prob),
            "train_accuracy": accuracy_score(y_train, y_train_pred),
            "train_precision": precision_score(y_train, y_train_pred, zero_division=0),
            "train_recall": recall_score(y_train, y_train_pred, zero_division=0),
            "train_f1": f1_score(y_train, y_train_pred, zero_division=0),

            "test_roc_auc": roc_auc_score(y_test, y_test_prob),
            "test_average_precision": average_precision_score(y_test, y_test_prob),
            "test_log_loss": log_loss(y_test, y_test_prob),
            "test_accuracy": accuracy_score(y_test, y_test_pred),
            "test_precision": precision_score(y_test, y_test_pred, zero_division=0),
            "test_recall": recall_score(y_test, y_test_pred, zero_division=0),
            "test_f1": f1_score(y_test, y_test_pred, zero_division=0),
        }

        mlflow.log_metrics(metricas)

        mlflow.log_params({
            "cv_best_roc_auc": grid.best_score_,
            "quantidade_features": X_train.shape[1],
            "quantidade_treino": X_train.shape[0],
            "quantidade_teste": X_test.shape[0],
            "taxa_churn_treino": float(y_train.mean()),
            "taxa_churn_teste": float(y_test.mean())
        })

        melhor_pipeline = grid.best_estimator_
        features_finais = melhor_pipeline.named_steps['onehot'].get_feature_names_out()
        mlflow.log_params({
            "features_finais": features_finais
        })

        # Registra os melhores parâmetros do grid
        for parametro, valor in grid.best_params_.items():
            mlflow.log_param(
                f"best_{parametro}",
                valor
            )

        mlflow.set_tags({
            'features': features_finais,
        })

        input_example = X_train.head(5)

        signature = infer_signature(
            X_train,
            melhor_estimador.predict_proba(X_train)
        )

        mlflow.sklearn.log_model(
            sk_model=melhor_estimador,
            artifact_path="modelo_validacao",
            signature=signature,
            input_example=input_example
        )

        resultados.append({
            "modelo": nome_modelo,
            "cv_roc_auc": grid.best_score_,
            **metricas,
            "melhores_parametros": grid.best_params_,
            "run_id": run.info.run_id
        })

        if metricas["test_roc_auc"] > melhor_roc_auc:
            melhor_roc_auc = metricas["test_roc_auc"]
            melhor_modelo_validacao = melhor_estimador
            melhor_nome_modelo = nome_modelo
            melhor_run_id = run.info.run_id
            melhores_parametros = grid.best_params_.copy()

# Comparação dos Modelos

In [0]:
# Comparação dos Modelos

df_resultados = pd.DataFrame(resultados)

df_resultados = df_resultados.sort_values(
    by=[
        "test_roc_auc",
        "test_average_precision"
    ],
    ascending=False
).reset_index(drop=True)

colunas_resultado = [
    "modelo",
    "cv_roc_auc",
    "test_roc_auc",
    "test_average_precision",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_log_loss"
]

print(df_resultados[colunas_resultado])

In [0]:
df_resultados.to_csv(
    "comparacao_modelos_churn.csv",
    index=False
)

# Retreino do melhor modelo com a base de Treino + Teste

In [0]:
# Refit do melhor modelo com treino + teste

X_completo = pd.concat([X_train, X_test], axis=0)

y_completo = pd.concat([y_train, y_test], axis=0)

modelo_producao = clone(melhor_modelo_validacao)

modelo_producao.fit(X_completo, y_completo)

print("Modelo vencedor:", melhor_nome_modelo)
print("Quantidade de registros no refit:", len(X_completo))
print("Melhores parâmetros:")
print(melhores_parametros)

# Registro do modelo final no MlFlow

In [0]:
# Registro do modelo final em produção no MlFlow

with mlflow.start_run(
    run_name=f"producao_{melhor_nome_modelo}"
) as run_producao:

    mlflow.set_tag(
        "tipo_execucao",
        "refit_treino_mais_teste"
    )

    mlflow.set_tag(
        "modelo_selecionado",
        melhor_nome_modelo
    )

    mlflow.set_tag(
        "run_origem_validacao",
        melhor_run_id
    )

    mlflow.log_params({
        "modelo_vencedor": melhor_nome_modelo,
        "quantidade_refit": len(X_completo),
        "quantidade_features": X_completo.shape[1],
        "taxa_churn_refit": float(y_completo.mean()),
        "criterio_selecao": "test_roc_auc",
        "roc_auc_obtido_no_teste": float(
            melhor_roc_auc
        )
    })

    for parametro, valor in melhores_parametros.items():
        mlflow.log_param(
            f"best_{parametro}",
            valor
        )

    # Registra as métricas históricas de seleção
    linha_vencedora = df_resultados.loc[
        df_resultados["modelo"] == melhor_nome_modelo
    ].iloc[0]

    mlflow.log_metrics({
        "validation_roc_auc": float(
            linha_vencedora["test_roc_auc"]
        ),
        "validation_average_precision": float(
            linha_vencedora["test_average_precision"]
        )
    })

    input_example = X_completo.head(5)

    assinatura_producao = infer_signature(
        X_completo,
        modelo_producao.predict_proba(X_completo)
    )

    model_info = mlflow.sklearn.log_model(
        sk_model=modelo_producao,
        artifact_path="modelo_producao",
        signature=assinatura_producao,
        input_example=input_example
    )

    run_id_producao = run_producao.info.run_id
